# 3.5 Bootstrap: implementation check on 2023
**Purpose.** Run the paired week-block bootstrap of paper Section 5.3 end to end on real scores before the test year. **The intervals below are an implementation check, not results**, because 2023 chose the XGBoost settings and every warning threshold.

**Method.** `frozen.compare`, the function the 2024 run calls. For each horizon and comparison (all XGBoost), it gives the AP difference on the 52 complete weeks and its 95% percentile interval. Weeks are counted from 1 January, so 31 December 2023 lies outside them, as 30 and 31 December do in 2024. Each repetition draws 52 weeks with replacement and weights every origin by the number of times its week was drawn, for both scores at once (paired) and with all districts of a week together. 2,000 repetitions use seed 1. A second seed (2) checks stability, and both runs are repeated with 10,000 repetitions if an endpoint moves by 0.005 or more.

**Inputs and outputs.** The 2023 scores of notebooks 3.0 to 3.3. Nothing is written, and 2024 is not read.

## 1. The 2023 scores and the week blocks
The score files are joined as in notebook 3.4, and the frozen design is checked first. Every origin must fall into one of the 52 weeks or onto 31 December.

In [1]:
import time

import numpy as np
import polars as pl
from sklearn.metrics import average_precision_score

from madrid_congestion import bootstrap
from madrid_congestion.config import HORIZONS, INTERIM_DATA_DIR, VALIDATION_YEAR
from madrid_congestion.frozen import CONTRASTS, check_frozen, compare, load_frozen

SOURCES = ["linear", "xgboost", "ablation", "sarima"]  # the score files of notebooks 3.0 to 3.3
KEYS = ["year", "minutes", "distrito", "slot_start"]
pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)
check_frozen()  # every frozen file present and consistent with the code

frames = [pl.read_parquet(INTERIM_DATA_DIR / f"scores-{source}.parquet").filter(pl.col("year") == VALIDATION_YEAR) for source in SOURCES]
scores = frames[0]
for source, df in zip(SOURCES[1:], frames[1:], strict=True):
    joined = scores.join(df, on=KEYS, how="full", validate="1:1", coalesce=True)
    assert joined.height == scores.height == df.height, source  # the same origins
    assert (joined["target"] == joined["target_right"]).all(), source  # the same targets
    scores = joined.drop("target_right")
by_horizon = {minutes: scores.filter(pl.col("minutes") == minutes).sort("distrito", "slot_start") for minutes in HORIZONS}

weeks = []
for minutes, df in by_horizon.items():
    block = bootstrap.week_block(df["slot_start"], VALIDATION_YEAR)
    outside = df.filter(block < 0)
    assert (outside["slot_start"].dt.date().cast(pl.String) == f"{VALIDATION_YEAR}-12-31").all()  # only 31 December
    per_week = df.filter(block >= 0).group_by(pl.Series("week", block[block >= 0])).agg(origins=pl.len(), positive=pl.col("target").sum())
    assert per_week.height == bootstrap.WEEKS
    weeks.append({
        "minutes": minutes, "origins": df.height, "outside_weeks": outside.height, "outside_positive": outside["target"].sum(),
        "origins_per_week_min": per_week["origins"].min(), "origins_per_week_max": per_week["origins"].max(),
        "positive_per_week_min": per_week["positive"].min(), "positive_per_week_max": per_week["positive"].max(),
    })
print(f"1 January {VALIDATION_YEAR} is a {by_horizon[30]['slot_start'].min():%A}; 52 weeks of seven days from it")
pl.DataFrame(weeks)

1 January 2023 is a Sunday; 52 weeks of seven days from it


minutes,origins,outside_weeks,outside_positive,origins_per_week_min,origins_per_week_max,positive_per_week_min,positive_per_week_max
i64,i64,i64,i64,i64,i64,i64,i64
30,482391,1246,2,6785,11247,0,318
60,461884,1174,4,6262,10961,0,639


## 2. Checks on the real scores
Four checks on the primary pair (direct and occupancy XGBoost) before any interval is read:
- with weight 1 inside the weeks, the bootstrap's weighted AP equals scikit-learn's AP;
- for one draw, the weighted AP equals scikit-learn's AP on a table in which every origin is repeated as often as its week was drawn;
- the same seed gives identical differences, and a score compared with itself gives exactly 0;
- the time of one repetition, to size the full run.

In [2]:
df = by_horizon[30]
target = df["target"].to_numpy()
block = bootstrap.week_block(df["slot_start"], VALIDATION_YEAR)
inside = block >= 0
first, second = CONTRASTS["primary: direct - occupancy"]
ranked = {name: bootstrap.RankedScore(target, df[name]) for name in (first, second)}

for name, score in ranked.items():
    own = score.ap(inside.astype(float))
    reference = average_precision_score(target[inside], df[name].to_numpy()[inside])
    assert np.isclose(own, reference, rtol=0, atol=1e-12), (name, own, reference)
    print(f"{name}: AP on the 52 weeks {own:.6f} = scikit-learn {reference:.6f}")

rng = np.random.default_rng(7)
draws = np.bincount(rng.integers(0, bootstrap.WEEKS, size=bootstrap.WEEKS), minlength=bootstrap.WEEKS)
weight = np.where(inside, draws[np.maximum(block, 0)], 0)
repeated = np.repeat(np.arange(df.height), weight)  # every origin as often as its week was drawn
for name, score in ranked.items():
    own = score.ap(weight.astype(float))
    reference = average_precision_score(target[repeated], df[name].to_numpy()[repeated])
    assert np.isclose(own, reference, rtol=0, atol=1e-12), (name, own, reference)
    print(f"{name}: weighted AP of one draw {own:.6f} = scikit-learn on {repeated.size:,} repeated origins {reference:.6f}")

once = bootstrap.paired_differences(target, df[first], df[second], block, 20, seed=1)
assert np.array_equal(once, bootstrap.paired_differences(target, df[first], df[second], block, 20, seed=1))
assert np.all(bootstrap.paired_differences(target, df[first], df[first], block, 20, seed=1) == 0)
start = time.time()
bootstrap.paired_differences(target, df[first], df[second], block, 100, seed=3)
print(f"same seed, same differences; a score against itself gives 0 | {(time.time() - start) / 100 * 1000:.0f} ms per repetition")

xgboost_direct: AP on the 52 weeks 0.720294 = scikit-learn 0.720294
xgboost_occupancy: AP on the 52 weeks 0.696210 = scikit-learn 0.696210
xgboost_direct: weighted AP of one draw 0.698433 = scikit-learn on 485,518 repeated origins 0.698433
xgboost_occupancy: weighted AP of one draw 0.675705 = scikit-learn on 485,518 repeated origins 0.675705
same seed, same differences; a score against itself gives 0 | 35 ms per repetition


## 3. The three comparisons at both horizons (check only)
`difference` is the point estimate on the 52 weeks, and `low` and `high` give the seed-1 interval. `repetitions` shows whether the second-seed check asked for 10,000, and `second_seed_move` is the largest change of an endpoint between the seeds. `difference_all_days` is the difference over every 2023 origin, as in paper Table 6.3.

In [3]:
start = time.time()
table = compare(by_horizon, VALIDATION_YEAR)
print(f"bootstrap of {table.height} comparisons in {(time.time() - start) / 60:.1f} minutes")
frozen_ap = load_frozen("thresholds-2023.json")
table = table.with_columns(
    difference_all_days=pl.Series([
        frozen_ap[str(row["minutes"])][CONTRASTS[row["comparison"]][0]]["ap_2023"] - frozen_ap[str(row["minutes"])][CONTRASTS[row["comparison"]][1]]["ap_2023"]
        for row in table.iter_rows(named=True)
    ]),
    width=pl.col("high") - pl.col("low"),
)
assert (table["low"] <= table["difference"]).all() and (table["difference"] <= table["high"]).all()
table.with_columns(pl.col("difference", "low", "high", "width", "second_seed_move", "difference_all_days").round(4))

bootstrap of 6 comparisons in 39.1 minutes


minutes,comparison,difference,low,high,repetitions,second_seed_move,difference_all_days,width
i64,str,f64,f64,f64,i64,f64,f64,f64
30,"""primary: direct - occupancy""",0.0241,0.0165,0.031,2000,0.0003,0.0241,0.0145
30,"""secondary: occupancy - flow""",0.163,0.149,0.1761,2000,0.0003,0.163,0.0271
30,"""secondary: with flow - without…",0.0042,0.0006,0.0076,2000,0.0003,0.0042,0.007
60,"""primary: direct - occupancy""",0.0244,0.0186,0.0298,2000,0.0001,0.0244,0.0112
60,"""secondary: occupancy - flow""",0.1246,0.112,0.1367,2000,0.0004,0.1246,0.0247
60,"""secondary: with flow - without…",0.0061,0.0029,0.01,2000,0.0001,0.0061,0.0071


## 4. Why pairing matters
A design check, not a result: the spread of the paired differences against the spread of each AP on its own over the same draws. If the two scores rise and fall together across weeks, their difference varies much less than either AP.

In [4]:
rows = []
for minutes, df in by_horizon.items():
    target = df["target"].to_numpy()
    block = bootstrap.week_block(df["slot_start"], VALIDATION_YEAR)
    for name, (a, b) in CONTRASTS.items():
        ra, rb = bootstrap.RankedScore(target, df[a]), bootstrap.RankedScore(target, df[b])
        rng = np.random.default_rng(1)
        ap_a, ap_b = [], []
        for _ in range(500):
            draws = np.bincount(rng.integers(0, bootstrap.WEEKS, size=bootstrap.WEEKS), minlength=bootstrap.WEEKS)
            weight = np.where(block >= 0, draws[np.maximum(block, 0)], 0).astype(float)
            ap_a.append(ra.ap(weight))
            ap_b.append(rb.ap(weight))
        ap_a, ap_b = np.array(ap_a), np.array(ap_b)
        rows.append({"minutes": minutes, "comparison": name, "sd_first": ap_a.std(), "sd_second": ap_b.std(),
                     "sd_difference": (ap_a - ap_b).std(), "correlation": np.corrcoef(ap_a, ap_b)[0, 1]})
pl.DataFrame(rows).with_columns(pl.exclude("minutes", "comparison").round(4))

minutes,comparison,sd_first,sd_second,sd_difference,correlation
i64,str,f64,f64,f64,f64
30,"""primary: direct - occupancy""",0.0098,0.0083,0.0038,0.926
30,"""secondary: occupancy - flow""",0.0083,0.0114,0.0071,0.7889
30,"""secondary: with flow - without…",0.0098,0.0098,0.0018,0.9837
60,"""primary: direct - occupancy""",0.0112,0.0102,0.0028,0.9699
60,"""secondary: occupancy - flow""",0.0102,0.013,0.0062,0.885
60,"""secondary: with flow - without…",0.0112,0.0119,0.0019,0.9886


**Result (implementation check, not a study result).** The bootstrap behaves as designed. Only the origins of 31 December fall outside the 52 weeks (1,246 at 30 minutes and 1,174 at 60 minutes, with 2 and 4 positive), and every week holds between 6,262 and 11,247 origins. All four checks pass. The six comparisons ran with 2,000 repetitions in 39 minutes, and the second seed moved no endpoint by more than 0.0004, so no rerun with 10,000 was needed. The point estimates on the 52 weeks agree with paper Table 6.3 to four decimals. Pairing works as intended: across draws, the two APs of a comparison have a correlation of 0.79 to 0.99, so the standard deviation of their difference is 1.2 to 5.9 times smaller than that of the less variable AP.

**Conclusion.** The 2024 run can call `compare` unchanged. The 2023 intervals are not reported as findings.